In [10]:
## 基本工具定义
from langchain.tools import tool

# 装饰器 -- 下面单引号包裹的内容(原来是解释函数的)变为工具的描述，帮助模型理解
# 工具名称(即函数名称)选择用蛇形命名法(snake_case)，即字母、数字字符、下划线和连字符
@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.
    
    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [11]:
## 自定义工具属性
### 自定义工具名称
@tool("web_search")     # 自定义名称
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)      # 输出 web_search

web_search


In [12]:
### 自定义工具描述(类似 skills 的触发规则)
@tool("calculator", description="Performs arithmetic calculators. Use this for math prol=blems.")
def calc(expression: str) -> str:
    """Evaluate mathmatical expressions."""
    return str(eval(expression))

**① 用户输入**  
你在 Claude Code 对话框发送：*"上海的天气怎么样？用摄氏度显示"*

**② 大模型读取工具说明书**  
Claude(模型) 分析意图后，查阅系统提示词中由 `@tool` 生成的 `WeatherInput` 类说明书（包含 `location` 必填、`units` 默认 `celsius` 等描述）。

**③ 大模型生成参数 JSON**  
Claude(模型) 根据说明书输出调用参数（省略了有默认值的字段）：  
`{"location": "上海"}`

**④ Agent 框架校验并实例化**  
后台捕获 JSON，用 `WeatherInput` 类验证，自动补全缺失的默认值：  
`WeatherInput(location="上海", units="celsius", include_forecast=False)`

**⑤ 框架调用你的函数**  
框架将 Pydantic 对象拆解，实际执行：  
`get_weather(location="上海", units="celsius", include_forecast=False)`

**⑥ 函数执行业务逻辑**  
- 判断 `units == "celsius"`，设 `temp = 22`  
- 拼接字符串（此时 `.upper()` 正确执行，取 `"celsius"[0].upper()` 得到 `"C"`）  
- `include_forecast` 为 `False`，不追加预报内容  
- 结果暂存为：`"Current weather in 上海: 22 degrees C"`

**⑦ 函数返回原始结果**  
`get_weather` 将上述字符串返回给 Agent 框架。

**⑧ 大模型翻译并回复用户**  
框架将字符串传给 Claude，Claude 重新组织成自然语言输出到你的聊天框：  
*"上海当前气温为 22 摄氏度。"*

In [13]:
## 高级模式自定义
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):          # 继承 创建数据模型的基类 BaseModel(让定义的 WeatherInput 类具备自动校验参数类型、必填项的能力)
    """Input for weather queries."""
    # location 为 str 类型的属性，Field 函数 为模型字段添加额外元数据，这里添加 description，让描述可以被大模型读取
    location: str = Field(
        description="City name or coordinates"  # 给大模型看的提示语，告诉模型“请在这里填城市名或坐标”
    )
    # Literal:精确限定值范围，这里明确限制 单位(units) 为摄氏度(celsius)或者华氏度(fahrenheit)
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", 
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecase"
    )

# 这个装饰器告诉底层的 AI 框架：“下面的 get_weather 函数是一个可被大模型调用的工具，传入参数的格式请严格按照 WeatherInput 这个类的定义来解析
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) ->str:   # 参数列表要与 WeatherInput 保持一致
    """Get current weather and optional forecase."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:    # 如果用户要求包含预报，就在结果末尾追加接下来五天的天气（这里硬编码为晴朗）
        result += "\nNext 5 days: Sunny"
    return result

## 保留参数名称 -- config 和 runtime 参数不能用于工具参数，否则会导致运行错误

In [14]:
# 访问上下文
## 1.短期记忆(状态): runtime: ToolRuntime
### 1.1 访问状态：runtime.state

# ToolRuntime类 是运行时上下文对象，提供对当前状态（state）、配置、回调等的访问，在工具函数内部可以通过参数注入获取
from langchain.tools import tool, ToolRuntime
# HumanMessage 类，是 LangChain 消息体系中的一种，表示来自用户（人类）的消息。在消息列表中，可以通过类型判断来区分用户消息、AI 消息或系统消息
from langchain.messages import HumanMessage

# 工具 1：从标准消息历史中提取最近的用户输入
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    # runtime.state 是工具执行时的状态字典，通常包含对话历史、用户偏好、上下文数据等；"messages" 是 LangChain 的标准键，存储了当前对话的所有消息列表（按时间顺序，最早在前）
    messages = runtime.state["messages"]
    
    # Find the last human message
    for message in reversed(messages):  # reversed 翻转，先取最近的信息
        if isinstance(message, HumanMessage):   # 有实例返回，即有最近的信息，直接返回其内容（一般为 str)
            return message.content
        
    return "No user messages found"

# Access custom state fields
# 工具 2：从自定义状态字段（用户偏好）中读取指定项
@tool
def get_user_preference(
    pref_name: str, 
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    # 从 runtime.state 中获取键为 "user_preferences" 的值，如果该键不存在则返回空字典 {}；"user_preferences" 是一个自定义状态字段，不是 LangChain 内置的，但开发者可以在外部将其注入到 state 中（例如通过 AgentExecutor 的 input 或 chain 的配置）
    preferences = runtime.state.get("user_preferences", {})
    # 从 preferences 字典中根据传入的 pref_name 键获取对应的值，不存在返回 str "Not set"
    return preferences.get(pref_name, "Not set")

**状态更新代码的流程案例**：

```text
[用户: "我叫小明"]
       ↓
[LLM → AIMessage(tool_calls=[set_user_name])]
       ↓
[LangGraph 调用工具]
   → new_name="小明"
   → runtime.tool_call_id="call_abc123"
       ↓
[工具返回 Command(update={...})]
       ↓
[LangGraph 原子性合并状态]
   → messages 追加 ToolMessage
   → user_name = "小明"
       ↓
[LLM 读取更新后状态 → "好的小明"]
```

In [15]:
### 1.2 更新状态：Command

# 导入 AgentState 类。这是 LangChain 代理（Agent）的基础状态类，通常作为构建自定义状态的父类，包含了 messages（消息列表）等核心字段
from langchain.agents import AgentState
# 导入 ToolMessage。这是 LangChain 消息体系中的一种特殊消息类型，专门用于返回工具（Tool）的执行结果。它必须包含一个 tool_call_id，用于关联触发这次调用的那条 ToolCall 请求
from langchain.messages import ToolMessage
# 用于定义可被代理(Agent)调用的工具
from langchain.tools import ToolRuntime, tool
# 关键：Command 是 LangGraph 中用于控制状态流转的特殊对象。当工具返回 Command 时，LangGraph 不会仅仅把返回值当作普通内容，而是会解析 Command 内部的指令（如 update、goto）来更新图状态（State）或跳转节点现
from langgraph.types import Command

class CustomState(AgentState):      # 继承弗雷 AgentState
    # 新增字段 user_name，意味着在整个对话图（Graph）的执行过程中，state 字典里除了标准的 messages 外，还会持久化保存一个 user_name 字段，且可以在不同节点间传递
    user_name: str                  
    
@tool
# runtime: ToolRuntime[None, CustomState]：这是泛型类型标注。第一个泛型参数 None 表示上下文（Context）为空；第二个泛型参数 CustomState 明确告诉类型检查器和框架，这个工具操作的状态是 CustomState 类型
# 返回值标注为 Command，表示该工具将通过返回指令来修改状态，而不是直接返回普通字符串
def set_user_name(new_name: str, runtime: ToolRuntime[None, CustomState]) -> Command:
    """Set the user's name in the conversation state."""
    # 构造并返回一个 Command 对象，核心在于 update 字典，它定义了要对全局状态执行的更新操作
    return Command(
        update={
            "user_name": new_name, 
            # LangGraph 默认对列表类型的更新采用追加或合并策略，具体取决于 reducer 函数，但标准 messages 字段配置为追加
            "messages": [
                ToolMessage(
                    content=f"User name set to {new_name}.",
                    # 重点：runtime.tool_call_id 是从运行时上下文中获取的本次工具调用的唯一 ID。LangGraph 要求 ToolMessage 必须对应一个先前存在的 ToolCall，通过这个 ID 将结果与请求配对，否则图执行会报错
                    tool_call_id=runtime.tool_call_id, 
                )
            ],
        }
    )

In [16]:
## 2. 上下文

# Python 标准库的 dataclass 装饰器，自动生成 __init__、__repr__ 等方法，用于快速定义类
from dataclasses import dataclass
# 加载环境变量，避免硬编码 API_KEY 暴露
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
# 导入高级 Agent 工厂函数。它封装了 LangGraph 的图构建、工具绑定、系统提示词注入等 boilerplate 代码
from langchain.agents import create_agent
# ToolRuntime：运行时上下文注入器，使工具能访问 Context 和 State 字段
from langchain.tools import tool, ToolRuntime

# 模拟的用户数据库字典。注意：这是全局只读数据源，不是 Graph State。实际项目中应替换为数据库查询或 API 调用
USER_DATABSE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000, 
        "email": "alice@example.com"
    }, 
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

# 定义上下文数据结构。仅包含 user_id 字段，标识当前会话所属用户。
# 使用 @dataclass 后，可直接通过 UserContext(user_id="user123") 构造实例，无需手写 __init__
@dataclass
class UserContext:
    user_id: str
    
@tool
# 定义工具函数。关键特征：没有 LLM 传入的业务参数，仅依赖 runtime。
# ToolRuntime[UserContext]：明确声明需要 UserContext 类型的上下文。LLM 调用此工具时无需传参，框架自动注入 context。
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    # 从运行时上下文中提取 user_id
    # 安全性体现：用户 ID 来自应用层注入的 Context，而非 LLM 生成的参数，彻底避免 LLM 幻觉或提示注入导致的越权访问
    user_id = runtime.context.user_id

    # 查询模拟数据库并格式化返回字符串
    # 返回值是普通 str，框架会自动将其包装为 ToolMessage 追加到消息列表。因为不需要修改 State，所以无需返回 Command
    if user_id in USER_DATABSE:
        user = USER_DATABSE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

# 加载环境变量，必须在模型定义前面
load_dotenv()

# 初始化 LLM 实例
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0.7
)

# 创建 Agent 实例
agent = create_agent(
    model, 
    tools=[get_account_info],       # 绑定可用工具 **列表**（可多个）
    context_schema=UserContext,     # 声明式契约，告知框架此 Agent 需要 UserContext，后续 invoke 时必须传入匹配的 context
    system_prompt="You are a financial assistant."
)

result = agent.invoke(              # invoke 调用
    # 第一个参数：初始 State（仅需提供用户消息）
    {"messages": [{"role": "user", "content": "What's my current balance?"}]}, 
    # 外部注入上下文。此值在整个图执行期间只读可用，不会被任何节点修改或持久化到 State 中
    context=UserContext(user_id="user123")      # Alice Johnson
)

# 格式化查看完整对话流
for msg in result["messages"]:
    role = type(msg).__name__           # HumanMessage / AIMessage / ToolMessage
    print(f"[{role}] {msg.content}")

[HumanMessage] What's my current balance?
[AIMessage] Sure! Let me check your account information.
[ToolMessage] Account holder: Alice Johnson
Type: Premium
Balance: $5000
[AIMessage] Your current account balance is **$5,000**. Here's a quick summary of your account:

- **Account Holder:** Alice Johnson
- **Account Type:** Premium
- **Balance:** $5,000

Is there anything else I can help you with?


In [ ]:
## 3. 长期记忆（存储）-- BaseStore(跨对话持久存储)
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
# ToolRuntime: 当工具函数签名中包含此参数时，LangGraph 会自动注入运行时上下文（包含 store、config 等），无需手动传参。这是工具访问 Store 的唯一标准入口。
from langchain.tools import tool, ToolRuntime
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

# 查找记忆
@tool 
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    # Docstring
    """查找用户信息。"""
    store = runtime.store
    #  Store 的命名空间（Namespace），类型为元组。用于逻辑隔离不同类别的数据（如 ("users",), ("config",)）;store.get(namespace, key): 返回 Item 对象或 None。必须通过 .value 获取实际存储的 Python 对象
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "未找到该用户"

# 更新记忆
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """保存用户信息"""
    store = runtime.store
    # store.put(namespace, key, value): 写入或覆盖数据。Key 必须是字符串，Value 可以是任意可序列化对象。
    store.put(("users",), user_id, user_info)
    return "成功保存用户信息。"

# 加载环境变量 API_KEY 和 BASE_URL
load_dotenv()

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    temperature=0.7
)

store = InMemoryStore()
agent = create_agent(
    model, 
    tools=[get_user_info, save_user_info], 
    # store=store: 将 Store 实例绑定到 Agent。同一个 store 对象是跨对话持久化的前提。如果每次创建新 Store，数据将无法共享。
    store=store
)

# 第一轮对话：保存用户信息
# Agent 解析自然语言 → 调用 save_user_info → 数据写入 store; 即使此次 invoke 结束，数据仍保留在 store 对象中。
agent.invoke({
    "messages": [{"role": "user", "content": "保存如下用户: user_id: abc123, 姓名: 张三, 年龄: 25, email: ZhangSan@langchain.dev"}]
})

# 第二轮对话：获取用户信息
# 全新的 messages 列表：证明没有依赖上一轮的对话历史; Agent 能查到数据，完全依赖于 store 的跨调用持久化能力。
result = agent.invoke({
    "messages": [{"role": "user", "content": "获取 id 为 'abc123' 的用户的信息"}]
})

# 格式化查看完整对话流
for msg in result["messages"]:
    role = type(msg).__name__           # HumanMessage / AIMessage / ToolMessage
    print(f"[{role}] {msg.content}")

[HumanMessage] 获取 id 为 'abc123' 的用户的信息
[AIMessage] 
[ToolMessage] {'姓名': '张三', '年龄': 25, 'email': 'ZhangSan@langchain.dev'}
[AIMessage] ID 为 `abc123` 的用户信息如下：

- **姓名**：张三
- **年龄**：25
- **邮箱**：ZhangSan@langchain.dev

如果您需要修改或保存其他信息，也可以告诉我。


In [18]:
## 4. 流写入器 -- runtime.stream_writer(使用它必须在 LangGraph 执行上下文内调用该工具)
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """获取给定城市的天气"""
    writer = runtime.stream_writer
    
    # 在工具执行时流式定制更新
    writer(f"寻找城市数据：{city}")
    writer(f"请求城市数据：{city}")
    
    return f"{city} 总是晴天!"
    

In [ ]:
## 5. 执行信息 -- runtime.execution_info -- 用途：日志审计、分布式追踪、根据重试次数调整策略。
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """记录执行身份信息。"""
    info = runtime.execution_info
    # thread_id: 会话唯一标识。用于区分不同用户的对话流。
    # run_id: 单次 invoke 调用的唯一标识。用于追踪特定请求的生命周期。
    # node_attempt: 关键调试字段。表示当前节点（Node）是第几次尝试执行。如果 Agent 因错误重试，该值会增加。可用于实现“指数退避”或记录失败次数。
    print(f"线程：{info.thread_id}，运行：{info.run_id}")
    print(f"重试状态：{info.node_attempt}")
    return "完成"

In [ ]:
## 6. 服务器信息 -- runtime.server_info -- 本地测试时若打印为空是正常的，部署到 LangGraph Platform 后才会生效。
from langchain.tools import tool, ToolRuntime

@tool
def get_assistant_scoped_data(runtime: ToolRuntime) -> str:
    """获取当前助理范围内的数据。"""
    server = runtime.server_info
    if server is not None:
        # assistant_id / graph_id: 识别当前运行的具体 Agent 实例和图结构版本。在多租户 SaaS 应用中，用于隔离不同客户的数据或配置。
        print(f"助理: {server.assistant_id}, 图: {server.graph_id}")
        if server.user is not None:
            # server.user: 获取经过身份验证的用户对象。可用于权限控制（如：只有管理员才能调用某些敏感工具）。
            print(f"已认真用户: {server.user.identity}")
    return "完成"

In [ ]:
# 工具节点 -- ToolNode(需要对工具执行模式进行精细控制的自定义工作流)
## 1. 基本用法
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

@tool
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"
@tool
def calculator(expression: str) -> str:
    """计算一个数学表达式"""
    return str(eval(expression))

# 用你的工具创建工具节点
# 核心作用：将工具列表封装为一个可执行的图节点。
# 自动匹配：当 LLM 返回包含 tool_calls 的 AIMessage 时，ToolNode 会自动根据 name 字段找到对应工具并执行。
# 批量执行：支持并行执行多个 tool_calls（默认行为），无需手动循环。
# 错误处理：内置异常捕获。若工具报错，不会中断图执行，而是生成一条包含错误信息的 ToolMessage 返回给 LLM，让其自我修正。
tool_node = ToolNode([search, calculator])

# 在一个图中使用
# MessagesState：ToolNode 要求状态必须包含 messages 键（List[BaseMessage]），它从中读取最后一条 AIMessage 的 tool_calls。
builder = StateGraph(MessagesState)
builder.add_node("tools", tool_node)

# 加入其他结点和边

In [ ]:
## 2. 工具返回值 -- string、object(对象)、Command
### 2.1 返回字符串（人类可读）-- 当工具应提供纯文本供模型读取并在其下一个响应中使用时，返回字符串
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """获取一个城市的天气"""
    return f"{city} 当前是雨天。"

# 返回值被转换为 ToolMessage（ToolMessage(content="北京 当前是雨天。")，这条消息被追加到 messages 列表中，成为 LLM 下一轮推理的上下文输入。）
# 模型看到该文本并决定下一步做什么 -- 假设你的 State 定义为 {"messages": [], "weather_cache": {}}，即使工具返回了天气信息，weather_cache 字段 依然为空，字符串返回值 没有能力 直接写入 weather_cache 或任何其他自定义状态字段。它只能影响 messages 列表（通过追加 ToolMessage）。
# 除非模型(如LLM 看到天气后，主动调用 save_weather(city, data) 工具来更新状态)或另一个工具稍后更改(Reducer/自定义逻辑，如在 ToolNode 后接一个自定义节点，解析 ToolMessage 内容并更新 State)，否则智能体状态字段不会更改

In [ ]:
### 2.2 返回对象 -- 当工具生成 模型应检查的结构化数据时，如 dict
from langchain.tools import tool

@tool
def get_weather_data(city: str) -> dict:
    """获取一个城市的结构化天气数据"""
    return {
        "城市": city,
        "温度(摄氏度)": 22,
        "状况": "雨天"
    }
    
# 该对象被序列化并作为工具输出返回; 模型可以读取特定字段并对其进行推理; 与字符串返回一样，这不会直接更新图状态
# 当后续推理受益于明确的字段而不是自由格式文本时使用此方法。

In [ ]:
### 2.3 返回命令 -- 当工具需要更新图状态时（例如，设置用户偏好或应用程序状态），返回一个 Command。可以返回一个带有或不带 ToolMessage 的 Command 。如果模型需要看到工具成功（例如，确认偏好更改），请在更新中包含一个 ToolMessage
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command

@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """设置回复语言的偏好"""
    # Command(update={...}): 这是一个显式的状态变更指令。LangGraph 在执行完该工具后，会立即将 update 字典中的内容合并到当前 State 中。
    return Command(
        update={
            # 1. 直接更新自定义状态字段
            "preferenced_language": language, 
            # 2. 同时向消息列表追加一条确认信息
            "messages": [
                # ToolMessage(...): 手动构造确认消息。注意：当使用 Command 时，LangGraph 不再自动生成 ToolMessage。如果你希望 LLM 知道操作成功了，必须手动把它放进 update["messages"] 里。
                ToolMessage(
                    content=f"语言设置为 {language}。",
                    # ⚠️ 关键：必须绑定原始 runtime.tool_call_id，绝对不能漏。LLM 通过此 ID 将 ToolMessage 与之前的 AIMessage(tool_calls) 配对。若缺失或错误，OpenAI/DeepSeek 等 API 会直接报错拒绝请求。
                    tool_call_id=runtime.tool_call_id
                )
            ], 
        }
    )
    
# 命令使用 update 更新状态；更新后的状态可供同一运行中的后续步骤使用；
# 对可能被并行工具调用更新的字段使用 reducer（归约器）。
# 问题场景：如果 LLM 在一次响应中调用了 3 个工具，且这 3 个工具都返回了 Command(update={"score": 10})，Without Reducer 会导致后一个覆盖前一个，最终 score=10 而非预期的 30。
# 解决方案：在 State 定义中使用 Annotated[int, operator.add]。这样 LangGraph 会自动将并行的 update 值累加。
# 当工具不仅返回数据，还修改智能体状态时使用此方法

In [28]:
## 3. 错误处理
from multiprocessing import Value

from langgraph.prebuilt import ToolNode

@tool
def search(query: str) -> str:
    """搜索工具"""
    return f"Results for {query}"

tools = [search]

# 默认：捕捉调用错误，重新抛出执行错误
tool_node = ToolNode(tools) 

# 步骤所有错误并返回错误信息给 LLM
tool_node = ToolNode(tools, handle_tool_errors=True)

# 自定义错误消息
tool_node = ToolNode(tools, handle_tool_errors="出错了，请重试!")

# 自定义错误操作者
def handle_error(e: ValueError) -> str:
    return f"Invalid input: {e}"

tool_node = ToolNode(tools, handle_tool_errors=handle_error)

# 只捕捉特定的异常类型
tool_node = ToolNode(tools, handle_tool_errors=(ValueError, TypeError))

In [ ]:
## 4. 使用 tools_condition 进行路由
from langgraph.prebuilt import ToolNode, tools_condition
# MessagesState: 内置状态类型，仅含 messages: Annotated[list, add_messages] 字段，自带消息追加 Reducer。
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.tools import tool
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv

load_dotenv()

# 定义工具
@tool
def search(query: str) -> str:
    """搜索网络信息"""
    return f"Results for: {query}"

# tools 列表：后续需同时传给 bind_tools() 和 ToolNode，两者必须一致，否则模型生成的 tool_call 名称在 ToolNode 中找不到对应实现。
tools = [search]

# 定义 LLM 调用结点
model = ChatDeepSeek(model="deepseek-v4-flash", temperature=0)

def call_llm(state: MessagesState):
    """绑定工具的 LLM 调用函数"""
    # bind_tools(tools): 将工具 schema 注入模型请求。这是 tools_condition 能工作的前提——未绑定时模型不会生成 tool_calls，图将直接终止。
    response = model.bind_tools(tools).invoke(state["messages"])
    return {"messages": [response]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")      # 入口固定指向 LLM
# tools_condition: 预构建的条件路由函数。它检查消息列表中最后一条 AIMessage 是否包含 tool_calls，有则返回 "tools"，无则返回 END。
builder.add_conditional_edges("llm", tools_condition)   # 动态路由，返回值为字符串，匹配已注册的节点名(tools 中)或 END
builder.add_edge("tools", "llm")    # 工具执行后回环，形成 ReAct 循环：推理→行动→观察→再推理

graph = builder.compile()

In [ ]:
## 5.状态注入
from langchain.tools import tool, ToolRuntime
from langgraph.prebuilt import ToolNode

@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """获取对话中信息的数量"""
    messages = runtime.state["messages"]
    return f"有 {len(messages)} 条信息"

tool_node = ToolNode([get_message_count])